# Apache Iceberg with PySpark — comprehensive examples

Based on **IceBergCatalogTest (1).ipynb**, using **Spark 3.5.x**, **Iceberg 1.10.0**,
**Hive Metastore**, and **HDFS**.

All examples use the fixed database **`iceberg.demo`** and table names such as
**`iceberg.demo.customers`**. SQL is written directly in `%%sql` or `spark.sql(...)` cells.
There are no generated database names, table-name variables, or SQL placeholder dictionaries.

Edit the Spark master hostname in section 2, then run the examples in order. The CREATE TABLE
cells expect empty training tables. If a table already exists, inspect it before using the
commented cleanup commands at the end to start again. Restarting the kernel does not reset tables.

Snapshot IDs are assigned by Iceberg, so the snapshot lessons save the actual IDs in simple
variables. These are used for time travel and rollback; table and database names stay fixed.

The notebook covers CRUD, MERGE, snapshots, time travel, rollback, branches/tags, schema and
partition evolution, writes, CDF, maintenance, streaming, and migration. Optional streaming,
file import, and file deletion remain disabled until you choose to run those cells.

## 1. Check Java and Spark

The cluster should have Spark 3.5.x, Scala 2.12, and the matching Iceberg 1.10.0 runtime JAR.
Run these shell commands on the notebook host. If PySpark is missing from the Python kernel,
install the same PySpark version as your cluster before starting Spark.

In [ ]:
!java -version
# !spark-submit --version
# !hdfs dfs -ls /user/hive/warehouse

Use your cluster's hostnames below. The `localhost` metastore and HDFS addresses match the
original notebook; change them if those services are on another host. Configure Hadoop XML
files and credentials in the cluster environment when required.

## 2. Start Spark with Hive Metastore and HDFS

Replace `your-spark-host` with your Spark master hostname. This assumes the Iceberg runtime
JAR is already installed. Restart the kernel when changing JARs or Spark extensions.
[Catalog configuration](https://iceberg.apache.org/docs/1.10.0/spark-configuration/).

In [ ]:
from pyspark.sql import SparkSession, functions as F
from decimal import Decimal

spark = (
    SparkSession.builder
    .appName("Iceberg-PySpark-Examples")
    .master("spark://your-spark-host:7077")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.iceberg.type", "hive")
    .config("spark.sql.catalog.iceberg.uri", "thrift://localhost:9083")
    .config("spark.sql.catalog.iceberg.warehouse", "hdfs://localhost:9000/user/hive/warehouse")
    .config("spark.sql.warehouse.dir", "hdfs://localhost:9000/user/hive/warehouse")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

## 3. Enable simple Spark SQL cells

Run this once to use `%%sql` in a Python notebook. It sends the cell directly to `spark.sql`.
Use one SQL statement per cell. For snapshot variables, use an ordinary Python f-string cell.

In [ ]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def sql(line, cell):
    spark.sql(cell).show(50, truncate=False)

## 4. Create the demo database

Use **`iceberg.demo`** throughout. `IF NOT EXISTS` reuses the same database.

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS iceberg.demo")
spark.sql("SHOW NAMESPACES IN iceberg").show(truncate=False)
spark.sql("SHOW TABLES IN iceberg.demo").show(truncate=False)

## 5. Customer table: format v2, decimals, properties

Iceberg is a table format over data files, not a database server. A commit publishes a new metadata
state atomically for one table. A snapshot describes a consistent set of data and delete files.
Use exact decimals for money. This table explicitly uses copy-on-write so the later changelog lab
has a controlled input history.

In [ ]:
%%sql
CREATE TABLE iceberg.demo.customers (
    customer_id BIGINT NOT NULL,
    name STRING,
    city STRING,
    amount DECIMAL(10,2)
) USING iceberg
TBLPROPERTIES (
    'format-version'='2',
    'write.format.default'='parquet',
    'write.parquet.compression-codec'='zstd',
    'write.update.mode'='copy-on-write',
    'write.delete.mode'='copy-on-write',
    'write.merge.mode'='copy-on-write'
)

In [ ]:
spark.sql('SHOW TABLES IN iceberg.demo').show(truncate=False)
spark.sql('DESCRIBE TABLE EXTENDED iceberg.demo.customers').show(truncate=False)
spark.sql('SHOW TBLPROPERTIES iceberg.demo.customers').show(truncate=False)

## 6. Insert the original sample and save its snapshot

Alice, Bob, and Carol are the starting data. Save the current snapshot ID and its timestamp
for the later recovery examples. These values come from this table's actual commit.

In [ ]:
spark.sql("""
INSERT INTO iceberg.demo.customers VALUES
    (1, 'Alice', 'Bangalore', CAST(1200 AS DECIMAL(10,2))),
    (2, 'Bob', 'Hyderabad', CAST(1800 AS DECIMAL(10,2))),
    (3, 'Carol', 'Chennai', CAST(950 AS DECIMAL(10,2)))
""")

initial_snapshot = spark.sql("""
    SELECT snapshot_id FROM iceberg.demo.customers.refs WHERE name = 'main'
""").first()["snapshot_id"]

initial_timestamp = spark.sql(f"""
    SELECT CAST(committed_at AS STRING)
    FROM iceberg.demo.customers.snapshots
    WHERE snapshot_id = {initial_snapshot}
""").first()[0]

initial_ms = spark.sql(f"""
    SELECT unix_millis(committed_at)
    FROM iceberg.demo.customers.snapshots
    WHERE snapshot_id = {initial_snapshot}
""").first()[0]

print("Initial snapshot:", initial_snapshot)
print("Initial timestamp:", initial_timestamp)
spark.sql("SELECT * FROM iceberg.demo.customers ORDER BY customer_id").show()

## 7. UPDATE and DELETE

After UPDATE Alice has 1500.00; after DELETE only IDs 1 and 2 remain.
These are new commits; historical files can remain reachable by older snapshots.

In [ ]:
spark.sql("""
UPDATE iceberg.demo.customers
SET amount = CAST(1500 AS DECIMAL(10,2))
WHERE customer_id = 1
""")
spark.sql("SELECT * FROM iceberg.demo.customers ORDER BY customer_id").show()

spark.sql("DELETE FROM iceberg.demo.customers WHERE customer_id = 3")
spark.sql("SELECT * FROM iceberg.demo.customers ORDER BY customer_id").show()

## 8. Deterministic MERGE / CDC ingestion

Source batches can contain multiple events for one key. Deduplicate by a complete, deterministic
ordering before MERGE; multiple source rows matching one target row can fail. Here the highest
sequence wins. A production pipeline should persist processed offsets or batch IDs as well.
Expected final IDs: 1, 2, 4; Alice's amount is 2200.00.

In [ ]:
from pyspark.sql.window import Window
incoming = spark.createDataFrame([
 (1, 'Alice Old Event', 'Bangalore', Decimal('2100.00'), 1, 'U'),
 (1, 'Alice Updated', 'Bangalore', Decimal('2200.00'), 2, 'U'),
 (4, 'David', 'Pune', Decimal('1750.00'), 1, 'I'),
], 'customer_id long, name string, city string, amount decimal(10,2), seq long, op string')
latest = (incoming.withColumn('rn', F.row_number().over(Window.partitionBy('customer_id').orderBy(F.desc('seq'))))
    .where('rn=1').drop('rn'))
latest.createOrReplaceTempView('customer_updates')
spark.sql("""MERGE INTO iceberg.demo.customers t USING customer_updates s ON t.customer_id=s.customer_id
 WHEN MATCHED AND s.op='D' THEN DELETE
 WHEN MATCHED AND s.op<>'D' THEN UPDATE SET t.name=s.name, t.city=s.city, t.amount=s.amount
 WHEN NOT MATCHED AND s.op<>'D' THEN INSERT (customer_id,name,city,amount)
 VALUES (s.customer_id,s.name,s.city,s.amount)""").show(truncate=False)
merged_snapshot = spark.sql("SELECT snapshot_id FROM iceberg.demo.customers.refs WHERE name='main'").first()['snapshot_id']
spark.sql('SELECT * FROM iceberg.demo.customers ORDER BY customer_id').show(truncate=False)

## 9. Metadata explorer

Start with snapshots and history, then drill into manifests and files. Metadata queries describe
physical layout; they are not substitutes for logical row counts when delete files exist.
Historical `all_*` tables may show the same file through multiple snapshots.
[Metadata query reference](https://iceberg.apache.org/docs/1.10.0/spark-queries/).

In [ ]:
spark.sql("SELECT * FROM iceberg.demo.customers.snapshots").show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.customers.history ORDER BY made_current_at").show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.customers.refs").show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.customers.metadata_log_entries").show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.customers.manifests").show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.customers.files").show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.customers.data_files").show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.customers.delete_files").show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.customers.partitions").show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.customers.entries").show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.customers.all_manifests").show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.customers.all_data_files").show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.customers.all_delete_files").show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.customers.all_entries").show(truncate=False)
spark.sql("SELECT customer_id, _file, _pos, _spec_id FROM iceberg.demo.customers").show(truncate=False)

## 10. Time travel by snapshot ID and timestamp

Time travel reads historical state without changing the current table. Expected: the initial read
includes Carol and Alice at 1200.00. Snapshot IDs are identifiers, not chronological counters.
Timestamp reads require retained snapshots and files. SQL uses the session's UTC time zone.

In [ ]:
spark.sql(f"""
SELECT * FROM iceberg.demo.customers
VERSION AS OF {initial_snapshot}
ORDER BY customer_id
""").show(truncate=False)

In [ ]:
spark.sql(f"""
SELECT * FROM iceberg.demo.customers
TIMESTAMP AS OF '{initial_timestamp}'
ORDER BY customer_id
""").show(truncate=False)

In [ ]:
historical = spark.read.format('iceberg').option('snapshot-id', str(initial_snapshot)).load('iceberg.demo.customers')
historical.show()
by_time = spark.read.format('iceberg').option('as-of-timestamp', initial_ms).load('iceberg.demo.customers')
by_time.show(truncate=False)

## 11. Compare snapshots as logical row sets

`exceptAll` preserves multiplicity. The two outputs are removed and added row images;
an updated row appears once in each. This comparison is useful even when physical rewrites occur,
but it scans both snapshots. Keep schemas aligned when comparing across schema evolution.

In [ ]:
before = spark.read.format('iceberg').option('snapshot-id', str(initial_snapshot)).load('iceberg.demo.customers')
after = spark.read.format('iceberg').option('snapshot-id', str(merged_snapshot)).load('iceberg.demo.customers')
print('Removed row images:')
before.exceptAll(after).show()
print('Added row images:')
after.exceptAll(before).show()

## 12. Rollback to an ancestor and restore a retained snapshot

Rollback changes the main reference and requires an ancestor. `set_current_snapshot` can select
a retained non-ancestor, which is how this lab restores the saved newer state after rollback.
Neither operation reverses schema changes. This exercise only changes the `iceberg.demo.customers` table.
[Recovery procedures](https://iceberg.apache.org/docs/1.10.0/spark-procedures/#rollback_to_snapshot).

In [ ]:
spark.sql(f"""
CALL iceberg.system.rollback_to_snapshot(
    table => 'demo.customers', snapshot_id => {initial_snapshot}
)
""").show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.customers ORDER BY customer_id").show()

spark.sql(f"""
CALL iceberg.system.set_current_snapshot(
    table => 'demo.customers', snapshot_id => {merged_snapshot}
)
""").show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.customers ORDER BY customer_id").show()

## 13. Rollback by timestamp

Use the saved timestamp of the first commit, then restore the saved merged snapshot.
Both operations act directly on `iceberg.demo.customers`.

In [ ]:
spark.sql(f"""
CALL iceberg.system.rollback_to_timestamp(
    table => 'demo.customers',
    timestamp => TIMESTAMP '{initial_timestamp}'
)
""").show(truncate=False)

spark.sql("SELECT * FROM iceberg.demo.customers ORDER BY customer_id").show()

spark.sql(f"""
CALL iceberg.system.set_current_snapshot(
    table => 'demo.customers', snapshot_id => {merged_snapshot}
)
""").show(truncate=False)

## 14. Tags and branches: isolated audit then publish

A tag pins a snapshot; a branch supports additional commits. Write to `branch_audit`, validate it,
then fast-forward main. Main must remain an ancestor of audit. Branches share the table schema;
they are not independent schemas. [Branch semantics](https://iceberg.apache.org/docs/1.10.0/branching/).

In [ ]:
spark.sql(f'ALTER TABLE iceberg.demo.customers CREATE TAG baseline AS OF VERSION {initial_snapshot} RETAIN 30 DAYS').show(truncate=False)
spark.sql(f'ALTER TABLE iceberg.demo.customers CREATE BRANCH audit AS OF VERSION {merged_snapshot} RETAIN 7 DAYS').show(truncate=False)
spark.sql("INSERT INTO iceberg.demo.customers.branch_audit VALUES (5, 'Eve', 'Delhi', CAST(2100 AS DECIMAL(10,2)))").show(truncate=False)
audit = spark.read.option('branch', 'audit').format('iceberg').load('iceberg.demo.customers')
audit.show()
spark.sql("CALL iceberg.system.fast_forward(table => 'demo.customers', branch => 'main', to => 'audit')").show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.customers VERSION AS OF 'baseline'").show(truncate=False)
spark.read.option('tag', 'baseline').format('iceberg').load('iceberg.demo.customers').show()
spark.sql('SELECT * FROM iceberg.demo.customers.refs').show(truncate=False)

## 15. Cherry-pick an append from a divergent branch

Both branches append different keys. Cherry-pick publishes the branch append onto current main;
it is not a general SQL MERGE or an arbitrary reconciliation of updates.

In [ ]:
spark.sql('ALTER TABLE iceberg.demo.customers CREATE BRANCH proposal').show(truncate=False)
spark.sql("INSERT INTO iceberg.demo.customers.branch_proposal VALUES (6, 'Farah', 'Mumbai', CAST(900 AS DECIMAL(10,2)))").show(truncate=False)
proposal_snapshot = int(spark.sql("SELECT snapshot_id FROM iceberg.demo.customers.refs WHERE name='proposal'").first()[0])
spark.sql("INSERT INTO iceberg.demo.customers VALUES (7, 'George', 'Kochi', CAST(800 AS DECIMAL(10,2)))").show(truncate=False)
spark.sql(f"CALL iceberg.system.cherrypick_snapshot(table => 'demo.customers', snapshot_id => {proposal_snapshot})").show(truncate=False)

## 16. Staged write-audit-publish (WAP ID)

This separate table demonstrates the older staged-snapshot workflow. Clear the session WAP ID
in `finally` so subsequent writes do not inherit staging. Do not combine WAP ID and WAP branch settings.
[WAP writes](https://iceberg.apache.org/docs/1.10.0/spark-writes/).

In [ ]:
spark.sql("""
CREATE TABLE iceberg.demo.wap_demo (id BIGINT)
USING iceberg TBLPROPERTIES ('format-version'='2')
""")
spark.sql("INSERT INTO iceberg.demo.wap_demo VALUES (1)")
spark.sql("ALTER TABLE iceberg.demo.wap_demo SET TBLPROPERTIES ('write.wap.enabled'='true')")

spark.conf.set('spark.wap.id', 'audit_demo')
try:
    spark.sql("INSERT INTO iceberg.demo.wap_demo VALUES (2)")
finally:
    spark.conf.unset('spark.wap.id')

# Main still contains only the first row.
spark.sql("SELECT * FROM iceberg.demo.wap_demo").show()
spark.sql("SELECT * FROM iceberg.demo.wap_demo.snapshots").show(truncate=False)

spark.sql("""
CALL iceberg.system.publish_changes(table => 'demo.wap_demo', wap_id => 'audit_demo')
""").show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.wap_demo").show()

## 17. Schema evolution and nested data

Iceberg tracks fields by ID. Renaming preserves identity; dropping and re-adding the same name
creates a new field. Supported promotions include int to long and decimal precision increases
with unchanged scale. These changes do not rewrite old files. Old records show null for added fields.
[Evolution](https://iceberg.apache.org/docs/1.10.0/evolution/).

In [ ]:
spark.sql("""CREATE TABLE iceberg.demo.schema_lab (
 id BIGINT NOT NULL, score INT, amount DECIMAL(10,2),
 profile STRUCT<city: STRING>, labels ARRAY<STRING>, attributes MAP<STRING,STRING>
) USING iceberg TBLPROPERTIES ('format-version'='2')""").show(truncate=False)
spark.sql("INSERT INTO iceberg.demo.schema_lab VALUES (1, 10, 12.50, named_struct('city','Pune'), array('new'), map('tier','gold'))").show(truncate=False)
spark.sql('ALTER TABLE iceberg.demo.schema_lab ADD COLUMNS (email STRING, profile.country STRING)').show(truncate=False)
spark.sql('ALTER TABLE iceberg.demo.schema_lab RENAME COLUMN score TO loyalty_points').show(truncate=False)
spark.sql('ALTER TABLE iceberg.demo.schema_lab ALTER COLUMN loyalty_points TYPE BIGINT').show(truncate=False)
spark.sql('ALTER TABLE iceberg.demo.schema_lab ALTER COLUMN amount TYPE DECIMAL(18,2)').show(truncate=False)
spark.sql('ALTER TABLE iceberg.demo.schema_lab ALTER COLUMN email AFTER id').show(truncate=False)
spark.sql('ALTER TABLE iceberg.demo.schema_lab DROP COLUMN email').show(truncate=False)
spark.sql('ALTER TABLE iceberg.demo.schema_lab ADD COLUMN email STRING').show(truncate=False)
spark.sql('SELECT * FROM iceberg.demo.schema_lab').show(truncate=False)

## 18. Identifier fields and write-time schema merge

Identifier fields describe row identity but do **not** enforce primary-key uniqueness.
Use required fields and validate duplicates in the ingestion pipeline. Automatic schema merge
is explicit here and confined to this lab table; review unexpected upstream fields before adopting it.

In [ ]:
spark.sql('ALTER TABLE iceberg.demo.schema_lab SET IDENTIFIER FIELDS id').show(truncate=False)
spark.sql("ALTER TABLE iceberg.demo.schema_lab SET TBLPROPERTIES ('write.spark.accept-any-schema'='true')").show(truncate=False)
evolved = spark.table('iceberg.demo.schema_lab').where('id=1').withColumn('id', F.lit(2).cast('long')).withColumn('source_system', F.lit('crm'))
evolved.writeTo('iceberg.demo.schema_lab').option('mergeSchema', 'true').append()
spark.sql('ALTER TABLE iceberg.demo.schema_lab DROP IDENTIFIER FIELDS id').show(truncate=False)
spark.sql("ALTER TABLE iceberg.demo.schema_lab UNSET TBLPROPERTIES ('write.spark.accept-any-schema')").show(truncate=False)

## 19. Hidden partitioning and partition evolution

Queries filter business columns; Iceberg derives partition pruning. Existing files retain their
old spec until rewritten. This lab changes daily partitioning to monthly and adds a bucket transform.
Use metadata `spec_id` to see mixed layouts; never infer the layout by manually scanning directories.
[Partitioning](https://iceberg.apache.org/docs/1.10.0/partitioning/).

In [ ]:
spark.sql("""CREATE TABLE iceberg.demo.events (event_id BIGINT, customer_id BIGINT, event_ts TIMESTAMP, region STRING, amount DECIMAL(12,2))
 USING iceberg PARTITIONED BY (days(event_ts)) TBLPROPERTIES ('format-version'='2')""").show(truncate=False)
spark.sql("INSERT INTO iceberg.demo.events VALUES (1,1,TIMESTAMP '2026-01-01 12:00:00','south',10.00), (2,2,TIMESTAMP '2026-01-02 13:00:00','north',20.00)").show(truncate=False)
spark.sql('ALTER TABLE iceberg.demo.events REPLACE PARTITION FIELD days(event_ts) WITH months(event_ts)').show(truncate=False)
spark.sql('ALTER TABLE iceberg.demo.events ADD PARTITION FIELD bucket(8, customer_id)').show(truncate=False)
spark.sql("INSERT INTO iceberg.demo.events VALUES (3,3,TIMESTAMP '2026-02-03 14:00:00','west',30.00)").show(truncate=False)
spark.sql('SELECT spec_id, partition, record_count FROM iceberg.demo.events.files').show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.events WHERE event_ts >= TIMESTAMP '2026-02-01 00:00:00' AND event_ts < TIMESTAMP '2026-03-01 00:00:00'").explain('formatted')

## 20. Sort order, distribution, and file sizes

Sort order governs new writes; it does not retroactively reorder files. Small teaching tables
cannot demonstrate production throughput. Tune using file sizes, scan plans, shuffle metrics,
and representative predicates. Target file size is a goal, not a guarantee.
[Tuning properties](https://iceberg.apache.org/docs/1.10.0/configuration/).

In [ ]:
spark.sql('ALTER TABLE iceberg.demo.events WRITE ORDERED BY region, event_ts').show(truncate=False)
spark.sql("ALTER TABLE iceberg.demo.events SET TBLPROPERTIES ('write.distribution-mode'='range', 'write.target-file-size-bytes'='134217728', 'write.metadata.metrics.default'='truncate(16)')").show(truncate=False)
spark.sql("INSERT INTO iceberg.demo.events VALUES (4,4,TIMESTAMP '2026-02-04 10:00:00','east',40.00)").show(truncate=False)
spark.sql('SELECT file_path, file_size_in_bytes, record_count, sort_order_id FROM iceberg.demo.events.files').show(truncate=False)

## 21. DataFrameWriterV2 and overwrite semantics

Use `writeTo` with catalog identifiers. Predicate overwrite replaces data matching a condition;
dynamic partition overwrite replaces partitions present in the incoming data. An unpartitioned
table is one partition, so dynamic overwrite there can replace the entire table. This separate
identity-partitioned table makes the expected boundary explicit.

In [ ]:
spark.sql("CREATE TABLE iceberg.demo.write_lab (id BIGINT, region STRING, value BIGINT) USING iceberg PARTITIONED BY (region) TBLPROPERTIES ('format-version'='2')").show(truncate=False)
spark.createDataFrame([(1,'south',10),(2,'north',20)], 'id long, region string, value long').writeTo('iceberg.demo.write_lab').append()
spark.createDataFrame([(3,'south',30)], 'id long, region string, value long').writeTo('iceberg.demo.write_lab').overwritePartitions()
spark.createDataFrame([(4,'north',40)], 'id long, region string, value long').writeTo('iceberg.demo.write_lab').overwrite(F.col('region') == 'north')
spark.sql('SELECT * FROM iceberg.demo.write_lab').show(truncate=False)

## 22. CTAS, RTAS, and table lifecycle

With SparkCatalog, CTAS and RTAS support atomic table creation/replacement. CTAS writes an
independent table; it is not a metadata-only clone. RTAS can also change schema and partitioning.
Use a dedicated derived table for this exercise. Rename support depends on the catalog.

In [ ]:
spark.sql("CREATE TABLE iceberg.demo.customer_summary USING iceberg TBLPROPERTIES ('format-version'='2') AS SELECT city, count(*) AS customers FROM iceberg.demo.customers GROUP BY city").show(truncate=False)
spark.sql("REPLACE TABLE iceberg.demo.customer_summary USING iceberg TBLPROPERTIES ('format-version'='2') AS SELECT city, count(*) AS customers, sum(amount) AS total_amount FROM iceberg.demo.customers GROUP BY city").show(truncate=False)
spark.sql('SELECT * FROM iceberg.demo.customer_summary').show(truncate=False)

## 23. Copy-on-write versus merge-on-read

COW rewrites affected data files. MOR can defer reconciliation to readers using delete files.
This v2 lab updates part of one coalesced file to make delete-file inspection useful. Spark typically
produces position deletes for these operations; equality deletes can originate from other writers.
Readers must support the table's delete representation. Physical record counts may exceed visible rows.

In [ ]:
spark.sql("""CREATE TABLE iceberg.demo.mor_lab (id BIGINT, value BIGINT) USING iceberg TBLPROPERTIES (
 'format-version'='2', 'write.update.mode'='merge-on-read',
 'write.delete.mode'='merge-on-read', 'write.merge.mode'='merge-on-read')""").show(truncate=False)
spark.range(20).selectExpr('id', 'id * 10 AS value').coalesce(1).writeTo('iceberg.demo.mor_lab').append()
spark.sql('UPDATE iceberg.demo.mor_lab SET value=999 WHERE id=1').show(truncate=False)
spark.sql('DELETE FROM iceberg.demo.mor_lab WHERE id=2').show(truncate=False)
spark.sql('SELECT content, file_path, record_count FROM iceberg.demo.mor_lab.files').show(truncate=False)
spark.sql('SELECT * FROM iceberg.demo.mor_lab.delete_files').show(truncate=False)

## 24. Incremental append reads

Incremental append scans use an exclusive start and inclusive end snapshot. They do not provide
general UPDATE/DELETE CDC. Use a dedicated append-only history so the example has unambiguous results.

In [ ]:
spark.sql("CREATE TABLE iceberg.demo.append_log (id BIGINT, message STRING) USING iceberg TBLPROPERTIES ('format-version'='2')").show(truncate=False)
spark.sql("INSERT INTO iceberg.demo.append_log VALUES (1,'first')").show(truncate=False)
append_start = spark.sql("SELECT snapshot_id FROM iceberg.demo.append_log.refs WHERE name='main'").first()['snapshot_id']
spark.sql("INSERT INTO iceberg.demo.append_log VALUES (2,'second'), (3,'third')").show(truncate=False)
append_end = spark.sql("SELECT snapshot_id FROM iceberg.demo.append_log.refs WHERE name='main'").first()['snapshot_id']
incremental = (spark.read.format('iceberg').option('start-snapshot-id', str(append_start))
    .option('end-snapshot-id', str(append_end)).load('iceberg.demo.append_log'))
incremental.show()

## 25. Changelog view with update images

Use the captured COW customer history before later branching. Update pairing uses `customer_id`.
Inspect change type and commit metadata; this is a bounded batch view, not a continuously running feed.
Delete-file support is version dependent, so do not substitute the MOR table into this lab.
[Changelog reference](https://iceberg.apache.org/docs/1.10.0/spark-procedures/#create_changelog_view).

In [ ]:
spark.sql(f"""CALL iceberg.system.create_changelog_view(
 table => 'demo.customers', changelog_view => 'customer_changes',
 options => map('start-snapshot-id','{initial_snapshot}','end-snapshot-id','{merged_snapshot}'),
 compute_updates => true, identifier_columns => array('customer_id'))""").show(truncate=False)
spark.sql('SELECT * FROM customer_changes ORDER BY _change_ordinal, customer_id, _change_type').show(truncate=False)

## 26. Compact data and manifests

Measure visible rows before and after compaction. Rewrites change physical layout and snapshots
while preserving logical content. The small lab explicitly forces rewrites; production jobs should
select worthwhile file groups. [Maintenance](https://iceberg.apache.org/docs/1.10.0/maintenance/).

In [ ]:
spark.sql("CALL iceberg.system.rewrite_data_files(table => 'demo.mor_lab', options => map('rewrite-all','true'))").show(truncate=False)
spark.sql("CALL iceberg.system.rewrite_position_delete_files(table => 'demo.mor_lab', options => map('rewrite-all','true'))").show(truncate=False)
spark.sql("CALL iceberg.system.rewrite_manifests(table => 'demo.events')").show(truncate=False)
spark.sql("CALL iceberg.system.rewrite_data_files(table => 'demo.events', strategy => 'sort', sort_order => 'region ASC NULLS LAST, event_ts ASC NULLS LAST', options => map('rewrite-all','true'))").show(truncate=False)
spark.sql('SELECT spec_id, count(*) AS files, sum(file_size_in_bytes) AS bytes FROM iceberg.demo.events.files GROUP BY spec_id').show(truncate=False)

## 27. Snapshot retention and orphan-file review

The fixed date below is an example retention cutoff; edit it for your lab. The orphan query
is a dry run. The expiration command is commented because it can remove historical snapshots
and their files. Retain enough history for readers, recovery, and streaming consumers.

In [ ]:
spark.sql("""
CALL iceberg.system.remove_orphan_files(
    table => 'demo.customers',
    older_than => TIMESTAMP '2026-01-01 00:00:00',
    dry_run => true
)
""").show(truncate=False)

# Optional: edit the cutoff before running snapshot expiration.
# spark.sql("""
# CALL iceberg.system.expire_snapshots(
#     table => 'demo.customers',
#     older_than => TIMESTAMP '2026-01-01 00:00:00',
#     retain_last => 5
# )
# """).show(truncate=False)

### Optional maintenance commands

```sql
-- Z-order rewrite:
CALL iceberg.system.rewrite_data_files(
  table => 'demo.events', strategy => 'sort',
  sort_order => 'zorder(customer_id,region)', options => map('rewrite-all','true'));

-- Remove references when no longer required:
ALTER TABLE iceberg.demo.customers DROP BRANCH audit;
ALTER TABLE iceberg.demo.customers DROP TAG baseline;
```

## 28. Scan planning and useful operational measurements

Compare plans and data-file metrics before changing tuning settings. Iceberg is not an index
service: partition pruning and column/file statistics reduce work when the predicates permit it.
Use Spark UI for actual task, spill, and shuffle measurements.

In [ ]:
spark.sql("SELECT event_id, amount FROM iceberg.demo.events WHERE region='west' AND customer_id=3").explain('formatted')
spark.sql("""SELECT count(*) AS data_files, sum(record_count) AS physical_records,
 round(avg(file_size_in_bytes)/1024,2) AS avg_kib,
 min(file_size_in_bytes) AS min_bytes, max(file_size_in_bytes) AS max_bytes
 FROM iceberg.demo.events.data_files""").show(truncate=False)
spark.sql('SELECT * FROM iceberg.demo.events.partitions').show(truncate=False)

## 29. Optimistic concurrency and isolation

Writers plan against a snapshot and validate at commit. Metadata commit retries do not make
every conflicting UPDATE or MERGE retry safe. On a validation conflict, reload/re-plan the business
operation and ensure its source batch is replayable. A table commit is atomic; two table writes are
not automatically a multi-table transaction. Identifier fields still do not enforce uniqueness.
[Reliability](https://iceberg.apache.org/docs/1.10.0/reliability/).

In [ ]:
spark.sql("""ALTER TABLE iceberg.demo.customers SET TBLPROPERTIES (
 'write.merge.isolation-level'='serializable',
 'write.update.isolation-level'='serializable',
 'write.delete.isolation-level'='serializable',
 'commit.retry.num-retries'='4')""").show(truncate=False)
spark.sql('SELECT customer_id, count(*) AS copies FROM iceberg.demo.customers GROUP BY customer_id HAVING count(*) > 1').show(truncate=False)

### Two-session concurrency exercise

Open another kernel against the same catalog and the **`iceberg.demo` namespace**. In both sessions,
read the same row, then issue overlapping MERGEs from separate source batches. A small SQL command
may commit too quickly to overlap: use larger disposable data to reproduce a validation conflict.
Record snapshot IDs and exceptions. Re-read the winner's state before retrying the losing business
operation. Do not blindly retry an increment, payment, or external side effect.

## 30. Optional bounded Structured Streaming write and read

The example uses fixed HDFS checkpoint paths. Set `RUN_STREAMING=True` to execute it.
Keep checkpoints when resuming the same stream. If you recreate the table, use fresh checkpoint
directories as well. Each query is stopped after the bounded demonstration.
[Streaming requirements](https://iceberg.apache.org/docs/1.10.0/spark-structured-streaming/).

In [ ]:
RUN_STREAMING = False

if RUN_STREAMING:
    spark.sql("""
    CREATE TABLE iceberg.demo.stream_events (event_ts TIMESTAMP, event_id BIGINT)
    USING iceberg TBLPROPERTIES ('format-version'='2')
    """)

    rate = spark.readStream.format('rate').option('rowsPerSecond', 3).load()
    query = (
        rate.selectExpr('timestamp AS event_ts', 'value AS event_id')
        .writeStream.format('iceberg').outputMode('append')
        .option('checkpointLocation', 'hdfs://localhost:9000/tmp/iceberg_demo/writer')
        .trigger(processingTime='5 seconds')
        .toTable('iceberg.demo.stream_events')
    )
    try:
        query.awaitTermination(20)
    finally:
        query.stop()

    reader = (
        spark.readStream.format('iceberg').load('iceberg.demo.stream_events')
        .writeStream.format('memory').queryName('iceberg_stream_observed')
        .option('checkpointLocation', 'hdfs://localhost:9000/tmp/iceberg_demo/reader')
        .trigger(availableNow=True).start()
    )
    try:
        if not reader.awaitTermination(60):
            raise TimeoutError('Streaming read did not finish within 60 seconds.')
        spark.sql('SELECT * FROM iceberg_stream_observed').show()
    finally:
        reader.stop()

### Streaming CDC with `foreachBatch`

For a CDC source, deduplicate each batch and MERGE into the target (section 8). `foreachBatch` can
replay a batch after failure; design idempotent assignments using source event sequence numbers.
Persist source progress with an approach that accounts for failure between state writes; merely
recording a batch ID in a second table is not atomic with the target MERGE. Append stream scans
do not emit a complete update/delete feed. Skipping delete/overwrite snapshots intentionally loses
those changes and is unsuitable for CDC correctness.

## 31. Optional Parquet file import

This uses the fixed HDFS directory below for new sample Parquet data. `add_files` registers
files without copying them. The input path must be unused for the first write. Do not remove
these files while the imported Iceberg table still references them.

In [ ]:
RUN_FILE_IMPORT = False

if RUN_FILE_IMPORT:
    spark.createDataFrame(
        [(1, 'legacy'), (2, 'archive')], 'id long, label string'
    ).write.mode('errorifexists').parquet('hdfs://localhost:9000/tmp/iceberg_demo/parquet_source')

    spark.sql("""
    CREATE TABLE iceberg.demo.imported_parquet (id BIGINT, label STRING)
    USING iceberg TBLPROPERTIES ('format-version'='2')
    """)

    spark.sql("""
    CALL iceberg.system.add_files(
        table => 'demo.imported_parquet',
        source_table => 'parquet.`hdfs://localhost:9000/tmp/iceberg_demo/parquet_source`'
    )
    """).show(truncate=False)

    spark.sql('SELECT * FROM iceberg.demo.imported_parquet').show()

## 32. Migration, registration, and catalog alternatives

These recipes require a separate compatible session/catalog and an existing source. They are
templates, not part of Run All. Verify ownership of shared files before retiring a source.

```sql
-- A SparkSessionCatalog wrapping spark_catalog is needed for in-place migration.
CALL spark_catalog.system.snapshot(
  source_table => 'legacy_db.parquet_table', table => 'legacy_db.iceberg_snapshot');
CALL spark_catalog.system.migrate(table => 'legacy_db.parquet_table');
-- Recover a catalog entry using the exact validated metadata JSON, not a data path:
CALL iceberg.system.register_table(
  table => 'recovery_db.recovered', metadata_file => 'hdfs://namenode/path/metadata/00042-uuid.metadata.json');
```

A migrated table and a snapshot-created table have different ownership implications. Avoid two
writable catalog entries pointing to the same metadata lineage. Snapshot/import workflows can
share data files, so purging one owner can damage another reader.

| Catalog | Session configuration concept | Infrastructure needed |
|---|---|---|
| Hive (default here) | `type=hive`, `uri=thrift://...` | Hive Metastore plus shared storage |
| Hadoop (alternative catalog) | `type=hadoop`, `warehouse=file:///...` | Filesystem with appropriate atomic operations |
| REST | `type=rest`, `uri=https://...` | REST catalog, its auth and storage configuration |
| Glue | `catalog-impl=org.apache.iceberg.aws.glue.GlueCatalog` | AWS bundle, IAM, Glue, S3FileIO |
| JDBC | `catalog-impl=org.apache.iceberg.jdbc.JdbcCatalog` | JDBC driver, database, credentials |
| Nessie | Vendor/version-specific catalog implementation | Nessie service and compatible client artifacts |

Configure credentials through your environment or secret provider. With S3, configure the matching
Iceberg AWS bundle and `org.apache.iceberg.aws.s3.S3FileIO`; HDFS configuration alone is insufficient.
Catalog-level versioning is distinct from Iceberg's table branches.

References: [Migration overview](https://iceberg.apache.org/docs/1.10.0/migration/),
[REST configuration](https://iceberg.apache.org/docs/1.10.0/spark-configuration/#catalog-configuration),
[AWS integration](https://iceberg.apache.org/docs/1.10.0/aws/).

## 33. Feature coverage and boundaries

| Feature area | Where / status |
|---|---|
| Catalogs, namespaces, DDL, properties | 2–5; alternate catalogs in 32 |
| INSERT, UPDATE, DELETE, MERGE | 6–8 |
| Snapshots, manifests, files, metadata columns | 9 |
| Snapshot/timestamp time travel, differences | 10–11 |
| Rollback, set current snapshot, timestamp recovery | 12–13 |
| Tags, branches, fast-forward, cherry-pick, WAP | 14–16 |
| Nested schema evolution and schema merge | 17–18 |
| Identifier fields (not enforced primary keys) | 18 |
| Hidden partitions, partition/spec evolution | 19 |
| Ordering, write distribution, file sizing, metrics | 20, 28 |
| WriterV2, dynamic/predicate overwrite, CTAS/RTAS | 21–22 |
| COW, MOR, position/equality delete concepts | 23 |
| Incremental append scans, changelog updates | 24–25 |
| Data/delete-file/manifest rewrite, sort, Z-order | 26–27 |
| Expiration, orphan dry run, reference retention | 27 |
| Optimistic concurrency and isolation | 29 |
| Streaming source/sink and CDC design | 30 |
| Parquet import, migration, registration | 31–32 |
| Parquet / ORC / Avro data files | Parquet labs; change `write.format.default` on a separate compatible table |
| Format v1 to v2 upgrade | Recipe below; upgrade is not a reversible toggle |
| Format v3, deletion vectors, row lineage, newer types | Separate runtime-specific curriculum; not asserted by these v2 labs |
| Views, materialized views, encryption, statistics/Puffin | Catalog/engine/API-dependent; not universally exposed by Spark 3.5 SQL |
| Multi-engine reads, REST auth, storage security | Require real external services and compatible readers |
| Multi-table transactions, uniqueness, secondary indexes | Not supplied automatically by these table operations |

**Optional isolated format upgrade:** create a new table with `'format-version'='1'`, append data,
then run `ALTER TABLE ... SET TBLPROPERTIES ('format-version'='2')`. Verify all readers before any
format upgrade. Changing a property does not retroactively rewrite existing files.

Do not enable format v3 on these labs just because a runtime accepts its property: validate deletion
vectors, row lineage, types, and every reader/writer against the precise release support matrix.
See the [Iceberg specification](https://iceberg.apache.org/spec/) for format-level definitions.

## 34. Troubleshooting

| Symptom | What to inspect |
|---|---|
| `ClassNotFoundException` for Iceberg | Matching runtime JAR present on driver/executors; restart kernel after changes |
| `NoSuchMethodError` / Scala linkage errors | Duplicate JARs or mismatched Spark minor / Scala binary version |
| SQL parser rejects CALL, MERGE, branch DDL | Iceberg extension installed before SparkContext; correct catalog and runtime |
| Metastore connection refused | URI, service status, firewall, hostname from the driver's network |
| HDFS failures in executor tasks | NameNode address, Hadoop config, permissions, network reachability |
| Catalog absent in SHOW CATALOGS | Initialize with SHOW NAMESPACES IN iceberg |
| Table already exists on cell rerun | Inspect the existing table; use the optional cleanup commands only to reset training data |
| Snapshot not found | Retention/expiration, wrong table, or ID copied from another run |
| MERGE multiple-source-row error | Deduplicate each source key with deterministic ordering |
| Concurrent validation conflict | Re-read and re-plan; check replay/idempotency guarantees |
| Too many tiny files | Trigger cadence, shuffle partitioning, distribution, compaction |
| SQL magic unknown | Execute section 3; use the Python kernel, not a SQL kernel |

## 35. Exercises with expected outcomes

1. Query `initial_snapshot` after the MERGE: Carol must exist and Alice must have 1200.00.
2. Compare initial and merged snapshots: Bob is unchanged; Carol is removed; David is added.
3. Inspect main before and after audit publication: Eve appears only after fast-forward.
4. Add a new schema field and read old rows: their new field is null without a file rewrite.
5. Inspect event `spec_id` before compaction: old and new partition layouts coexist.
6. Compare MOR logical rows before and after compaction: the row set is identical.
7. Run the duplicate-key query: no rows should be returned for the customer lab.
8. Inspect the tables in `iceberg.demo`: their names stay the same across kernel restarts.

## 36. Review the results

Customers should contain IDs 1, 2, 4, 5, 6, and 7 after the complete customer/branch lessons.
The event table should have four rows, and the MOR table nineteen.

In [ ]:
spark.sql("SHOW TABLES IN iceberg.demo").show(truncate=False)
spark.sql("SELECT * FROM iceberg.demo.customers ORDER BY customer_id").show()
spark.sql("SELECT COUNT(*) AS event_count FROM iceberg.demo.events").show()
spark.sql("SELECT COUNT(*) AS mor_count FROM iceberg.demo.mor_lab").show()

## 37. Optional cleanup

These commands are commented. Run only the individual commands for training tables you want
to delete. `PURGE` removes the table's files and historical snapshots. The shared `demo`
database is kept. Imported Parquet data and streaming checkpoints are not removed here.

In [ ]:
# spark.sql("DROP TABLE IF EXISTS iceberg.demo.customers PURGE")
# spark.sql("DROP TABLE IF EXISTS iceberg.demo.wap_demo PURGE")
# spark.sql("DROP TABLE IF EXISTS iceberg.demo.schema_lab PURGE")
# spark.sql("DROP TABLE IF EXISTS iceberg.demo.events PURGE")
# spark.sql("DROP TABLE IF EXISTS iceberg.demo.write_lab PURGE")
# spark.sql("DROP TABLE IF EXISTS iceberg.demo.customer_summary PURGE")
# spark.sql("DROP TABLE IF EXISTS iceberg.demo.mor_lab PURGE")
# spark.sql("DROP TABLE IF EXISTS iceberg.demo.append_log PURGE")
# spark.sql("DROP TABLE IF EXISTS iceberg.demo.stream_events PURGE")
# spark.stop()

## References and validation status

The authored examples target the supplied Spark 3.5 / Iceberg 1.10.0 environment. Consult the pinned
[Spark DDL](https://iceberg.apache.org/docs/1.10.0/spark-ddl/),
[writes](https://iceberg.apache.org/docs/1.10.0/spark-writes/),
[queries](https://iceberg.apache.org/docs/1.10.0/spark-queries/),
[procedures](https://iceberg.apache.org/docs/1.10.0/spark-procedures/), and
[streaming](https://iceberg.apache.org/docs/1.10.0/spark-structured-streaming/) documentation.

Notebook structure and transformed Python cell syntax are checked by the companion builder.
No Spark execution outputs are prepopulated. The authoring environment does not have PySpark or
the supplied Hive/HDFS cluster, so end-to-end execution must be performed in your configured lab.
The expected results in the explanations have not been verified on your cluster here.